In [2]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)

# Change the Drive letter to "F" or "D" depending on the mounted SSD drive in the PC

#======================================================================================================================
# Select this path  for training data updates
#FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Anubis")

# Select this path  for testing data
FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Anubis")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 14 matches to update. Processing...

Successfully updated: 6ff03e54-e7b1-4056-b6b1-ad53c786458f
Successfully updated: 5af177fd-a53d-48f5-90f3-7b4cb1800d0b
Successfully updated: 377699e6-47ca-427c-8a5f-b2ff3a1537d3
Successfully updated: a9c0ffbd-159f-4d34-bdf2-74286dd9c3fd
Successfully updated: e31a731c-fcc2-4a25-bdc2-39db036cfde4
Successfully updated: 0a508e1b-80ac-4dc8-88dd-b6f0e627e682
Successfully updated: 77ab23d0-ffd1-486c-9963-c372b8a200ef
Successfully updated: 84c0b5a1-7ae8-476b-9b1d-568598950716
Successfully updated: d4a8f729-03f0-4dff-8a51-9ba5734f3715
Successfully updated: 4cf091d8-29ac-4e27-a757-d108aed92200
Successfully updated: c573f64c-fc43-444b-9c7f-01ee4f64cedf
Successfully updated: 1c7928cb-a9d8-451b-a59e-b54d3edd726d
Successfully updated: bdfd6963-c329-4708-b3f4-bf34275697cc
Successfully updated: f0172964-13c5-4473-9bb0-7850042ee6a7

 UPDATE COMPLETE: 14/14 files modified.
